In [ ]:
%load_ext autoreload
%autoreload 2
# %flow mode reactive

# TODO: there will probably be a number of unecessary imports here, clean it up later

import datetime
import sys
import os
import warnings
from IPython.display import HTML
from pathlib import Path
from typing import Any, Tuple, List, Dict

import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objs as go
import statsmodels.api as sm
from plotly.subplots import make_subplots
from tqdm.notebook import tqdm

import datajoint as dj
from aeon.dj_pipeline.analysis.block_analysis import *
from aeon.dj_pipeline import acquisition, streams, subject
from swc.aeon.io import api as aeon_api
from aeon.schema.schemas import social02

cwd = os.getcwd()
project_path = os.path.join(cwd, "ProjectAeon", "aeon_scratchpad", "aeon_analysis", "aeon_methods_paper")
sys.path.append(project_path)
from data_io_utils import save_all_experiment_data, load_data_from_parquet

## Introduction

In this notebook you'll be working with behavioral data collected from experiments in which two mice naturally learned to forage for food in an environment with three patches whose reward rates changed dynamically over time.

The experiments consists of three phases each:

1. A "presocial" phase, where each mouse is in the environment alone for 3-4 days.
2. A "social" phase, where the two mice are in the environment together for 2 weeks.
3. A "post-social" phase, where each mouse is in the environment alone again for 3-4 days.

The goal of the experiment was to understand how mouse behavior changes as they learn to forage for food in the environment, and how their behavior differs in social vs. solo settings.

<img src="./tricentre_hackathon_assets/example_foraging_over_blocks.png" width="100%">

<img src="./tricentre_hackathon_assets/social02_env_protocol.png" width="100%">

<img src="./tricentre_hackathon_assets/social02_exp_timelines.png" height="300">

Maybe a way to embed mermaid diagram directly, but couldn't get it to work.

<script src="https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js"></script>
<script>mermaid.initialize({startOnLoad:true});</script>

<div class="mermaid">
%%{init: {'theme': 'dark'}}%%
gantt
    title Social0.2
    dateFormat  YYYY-MM-DD
    
    section Aeon3
    BAA-1104045    :2024-01-31, 2024-02-03
    Clean    :2024-02-04, 2024-02-05
    BAA-1104047    :2024-02-05, 2024-02-08
    Clean    :2024-02-08, 2024-02-09
    Tube Test    :2024-02-09, 2024-02-10
    BAA-1104045 + BAA-1104047    :2024-02-09, 2024-02-23
    Clean    :2024-02-23, 2024-02-24
    BAA-1104045    :2024-02-25, 2024-02-28
    Clean    :2024-02-28, 2024-02-29
    BAA-1104047    :2024-02-28, 2024-03-02
    
    section Aeon4
    BAA-1104048    :2024-01-31, 2024-02-03
    Clean    :2024-02-04, 2024-02-05
    BAA-1104049    :2024-02-05, 2024-02-08
    Clean    :2024-02-08, 2024-02-09
    Tube Test    :2024-02-09, 2024-02-10
    BAA-1104048 + BAA-1104049    :2024-02-09, 2024-02-23
    Clean    :2024-02-23, 2024-02-24
    BAA-1104048    :2024-02-25, 2024-02-28
    Clean    :2024-02-28, 2024-02-29
    BAA-1104049    :2024-02-28, 2024-03-02
</div>

In [ ]:
data_dir = Path("/ceph/aeon/aeon/code/scratchpad/methods_paper_data")
os.makedirs(data_dir, exist_ok=True)
cm2px = 5.2  # 1 cm = 5.2 px roughly in aeon arenas

In [ ]:
experiments = [
    {"name": "social0.2-aeon3", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 16:00:00', "social_end": '2024-02-23 13:00:00', "postsocial_start": '2024-02-25 17:00:00', "postsocial_end": '2024-03-02 14:00:00'},
    {"name": "social0.2-aeon4", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 17:00:00', "social_end": '2024-02-23 12:00:00', "postsocial_start": '2024-02-25 18:00:00', "postsocial_end": '2024-03-02 13:00:00'},
]

In [ ]:
# TODO: check this is necessary, hopefully already ingested in DJ
def clean_swaps(df: pd.DataFrame) -> pd.DataFrame:
    """
    Fast swap‐correction that returns the original position df's columns,
    but with x,y replaced by the cleaned tracks.

    Steps:
      0) sort & get identities
      1) reset_index so we can merge back later
      2) pivot x,y into 2×T arrays
      3) prepare cleaned arrays
      4) find first fully‐observed column and initialize
      5) loop over t=first_i+1..T-1:
            • swap‐correction logic (same as before)
            • **immediately** update track_votes whenever x_raw[:,t] & x_clean[:,t] are both finite
      6) decide final swap based on accumulated votes
      7) rebuild cleaned‐coord DataFrame & merge back
    """

    # 0) sort & get identities
    df = df.sort_index()
    ids = df['identity_name'].unique()
    assert len(ids) == 2, "Need exactly two identities"

    # 1) reset_index so we can merge back later
    df2 = df.reset_index()
    time_col = df2.columns[0]  # timestamp column name

    # 2) pivot x,y into 2×T arrays
    wide = df2.pivot(index=time_col, columns='identity_name', values=['x','y'])
    times = wide.index.values
    T = len(times)
    x_raw = np.vstack([wide['x'][ids[0]].values,
                       wide['x'][ids[1]].values])
    y_raw = np.vstack([wide['y'][ids[0]].values,
                       wide['y'][ids[1]].values])

    # 3) prepare cleaned arrays
    x_clean = np.full_like(x_raw, np.nan)
    y_clean = np.full_like(y_raw, np.nan)

    # 4) find first fully‐observed column
    valid = np.isfinite(x_raw).all(axis=0)
    first_i = np.argmax(valid)
    last_x = x_raw[:, first_i].copy()
    last_y = y_raw[:, first_i].copy()
    x_clean[:, first_i] = last_x
    y_clean[:, first_i] = last_y

    # Initialize vote‐matrix: [cleaned_track, original_identity]
    track_votes = np.zeros((2, 2), dtype=np.int64)

    # If the very first frame is fully observed, count those votes now:
    if valid[first_i]:
        # At t=first_i, x_clean[:,first_i] == x_raw[:,first_i], so track 0 ← orig 0, track 1 ← orig 1
        track_votes[0, 0] += 1
        track_votes[1, 1] += 1

    # 5) loop and swap‐correct *and* vote in one pass
    for t in tqdm(range(first_i + 1, T), desc="Cleaning frames"):
        # 5a) if not all raw‐points are finite, just carry‐forward
        if not np.isfinite(x_raw[:, t]).all():
            for k in (0, 1):
                if np.isfinite(x_raw[k, t]):
                    x_clean[k, t] = last_x[k]
                    y_clean[k, t] = last_y[k]
            # No "vote" in incomplete frames
            continue

        # 5b) if the two mice are very close, keep original order
        inter_mouse_dist = np.hypot(x_raw[0, t] - x_raw[1, t],
                                    y_raw[0, t] - y_raw[1, t])
        if inter_mouse_dist < 100:
            x_clean[:, t] = x_raw[:, t]
            y_clean[:, t] = y_raw[:, t]
            # immediate vote: clean track 0 came from raw‐0, clean 1 from raw‐1
            track_votes[0, 0] += 1
            track_votes[1, 1] += 1
            last_x = x_raw[:, t].copy()
            last_y = y_raw[:, t].copy()
            continue

        # 5c) compute distances to previous frame to decide swap
        dx = x_raw[:, t][:, None] - last_x[None, :]
        dy = y_raw[:, t][:, None] - last_y[None, :]
        dist = np.hypot(dx, dy)
        d_same = dist[0, 0] + dist[1, 1]
        d_swap = dist[0, 1] + dist[1, 0]

        # 5d) if even the best assignment is too big, carry‐forward
        if min(d_same, d_swap) > 90:
            x_clean[:, t] = last_x
            y_clean[:, t] = last_y
            # No vote, because x_clean did not actually come from x_raw this frame
            continue

        # 5e) pick same vs swapped assignment
        if d_same <= d_swap:
            x_clean[:, t] = x_raw[:, t]
            y_clean[:, t] = y_raw[:, t]
            # vote: clean₀ came from raw₀, clean₁ from raw₁
            track_votes[0, 0] += 1
            track_votes[1, 1] += 1
        else:
            x_clean[:, t] = x_raw[::-1, t]
            y_clean[:, t] = y_raw[::-1, t]
            # vote: clean₀ came from raw₁, clean₁ from raw₀
            track_votes[0, 1] += 1
            track_votes[1, 0] += 1

        # 5f) update last_x/last_y
        last_x = x_clean[:, t].copy()
        last_y = y_clean[:, t].copy()

    # 6) Final identity swap based on majority vote
    need_swap = track_votes[0, 1] > track_votes[0, 0]
    if need_swap:
        print(f"Swapping final tracks based on SLEAP majority vote")
        print(f"Track votes:\n{track_votes}")
        x_clean = x_clean[::-1, :]
        y_clean = y_clean[::-1, :]

    # 7) build cleaned coord table, naming columns x,y
    cleaned = pd.DataFrame({
        time_col: np.repeat(times, 2),
        'identity_name': np.tile(ids, T),
        'x': x_clean.ravel(order='F'),
        'y': y_clean.ravel(order='F'),
    })

    # 8) drop the old x,y and merge back everything else
    df2_noxy = df2.drop(columns=['x', 'y'])
    result = (
        df2_noxy
        .merge(cleaned, on=[time_col, 'identity_name'], how='right')
        .set_index(time_col)
        .sort_index()
    )

    return result


## Patch data

In [ ]:
def load_subject_patch_data(
    key: dict[str, str],
    period_start: str,
    period_end: str
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Loads subject patch data for a specified time period.

    Args:
        key (dict): The key to filter the subject patch data.
        period_start (str): The start time for the period.
        period_end (str): The end time for the period.

    Returns:
        tuple: A tuple containing:
            - patch_info (pd.DataFrame): Information about patches.
            - block_subject_patch_data (pd.DataFrame): Data for the specified period.
            - block_subject_patch_pref (pd.DataFrame): Preference data for the specified period.
    """
    patch_info = (
        BlockAnalysis.Patch()
        & key
        & f"block_start >= '{period_start}'"
        & f"block_start <= '{period_end}'"
    ).fetch(
        "block_start", "patch_name", "patch_rate", "patch_offset", "wheel_timestamps", as_dict=True
    )

    block_subject_patch_data = (
        BlockSubjectAnalysis.Patch()
        & key
        & f"block_start >= '{period_start}'"
        & f"block_start <= '{period_end}'"
    ).fetch(format="frame")

    block_subject_patch_pref = (
        BlockSubjectAnalysis.Preference()
        & key
        & f"block_start >= '{period_start}'"
        & f"block_start <= '{period_end}'"
    ).fetch(format="frame")

    if patch_info:
        patch_info = pd.DataFrame(patch_info)

    if not block_subject_patch_data.empty:
        block_subject_patch_data.reset_index(inplace=True)

    if not block_subject_patch_pref.empty:
        block_subject_patch_pref.reset_index(inplace=True)

    return patch_info, block_subject_patch_data, block_subject_patch_pref

In [ ]:
def ensure_ts_arr_datetime(array):
    if len(array) == 0:
        return np.array([], dtype='datetime64[ns]')
    else:
        return np.array(array, dtype='datetime64[ns]')

In [ ]:
patch_info_dict = {}
subject_patch_data_dict = {}
subject_patch_pref_dict = {}

for exp in experiments:
    key = {"experiment_name": exp["name"]}

    # Define periods
    periods = {
        "presocial": (exp["presocial_start"], exp["presocial_end"]),
        "social": (exp["social_start"], exp["social_end"]),
        "postsocial": (exp["postsocial_start"], exp["postsocial_end"])
    }

    # Initialize nested dictionaries for this experiment
    patch_info_dict[exp["name"]] = {}
    subject_patch_data_dict[exp["name"]] = {}
    subject_patch_pref_dict[exp["name"]] = {}

    # Load data for each period
    for period_name, (period_start, period_end) in periods.items():
        period_start = datetime.strptime(period_start, "%Y-%m-%d %H:%M:%S")
        period_end = datetime.strptime(period_end, "%Y-%m-%d %H:%M:%S")

        # Load data for this period
        patch_info, block_subject_patch_data, block_subject_patch_pref = (
            load_subject_patch_data(key, period_start, period_end)
        )

        # Drop nans for 'final_preference' columns
        block_subject_patch_pref = block_subject_patch_pref.dropna(subset=
            ["final_preference_by_time", "final_preference_by_wheel"]
        )

        # Extra processing on patch_info
        if not patch_info.empty:

            # Add experiment_name and period columns for reference
            patch_info.insert(0, "experiment_name", exp["name"])
            patch_info.insert(1, "period", period_name)

        # Extra processing on block_subject_patch_data
        if not block_subject_patch_data.empty:

            # Add period column for reference
            block_subject_patch_data.insert(1, "period", period_name)

            # For pre-social and post-social periods check n_subjects per block (should == 1)
            if period_name in ["presocial", "postsocial"]:
                n_subjects = (
                    block_subject_patch_data.groupby("block_start")["subject_name"].nunique()
                )
                if (n_subjects != 1).any():
                    warnings.warn(
                        f"Pre or post social data for {exp['name']} has blocks with more than one "
                        f"subject being tracked. Data needs to be fixed or cleaned."
                    )

        if not block_subject_patch_pref.empty:
            # Add period column for reference
            block_subject_patch_pref.insert(1, "period", period_name)

            # Ensure timestamps are correct type (datetime64[ns])
            if 'pellet_timestamps' in block_subject_patch_data.columns:
                block_subject_patch_data['pellet_timestamps'] = block_subject_patch_data['pellet_timestamps'].apply(ensure_ts_arr_datetime)

            if 'in_patch_rfid_timestamps' in block_subject_patch_data.columns:
                block_subject_patch_data['in_patch_rfid_timestamps'] = block_subject_patch_data['in_patch_rfid_timestamps'].apply(ensure_ts_arr_datetime)

            if 'in_patch_timestamps' in block_subject_patch_data.columns:
                block_subject_patch_data['in_patch_timestamps'] = block_subject_patch_data['in_patch_timestamps'].apply(ensure_ts_arr_datetime)

        # Store the data (patch_info as DataFrame now)
        patch_info_dict[exp["name"]][period_name] = patch_info
        subject_patch_data_dict[exp["name"]][period_name] = block_subject_patch_data
        subject_patch_pref_dict[exp["name"]][period_name] = block_subject_patch_pref

In [ ]:
# TODO: Check this works
# TODO: Are there columns we need to explain/units we need to clarify?
exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Extract example DataFrames
df1 = patch_info_dict[exp_name][period]
df2 = subject_patch_data_dict[exp_name][period]
df3 = subject_patch_pref_dict[exp_name][period]

# Display the first few rows of each
print(f"Patch Info — {exp_name}, {period}")
display(df1.head())

print(f"\nSubject Patch Data — {exp_name}, {period}")
display(df2.head())

print(f"\nSubject Patch Preference — {exp_name}, {period}")
display(df3.head())

In [ ]:
# TODO: add example plots/uses for the patch_info and subject_patch_pref DataFrames

In [ ]:
# TODO: Check this works
exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Get the subject patch data
df = subject_patch_data_dict[exp_name][period]

# Ensure data is sorted and ready
dt_seconds = 0.02
subjects = sorted(df['subject_name'].unique())
patches = sorted(df['patch_name'].unique())

fig = go.Figure()

# Build each trace (continuous + Δ>0.5 downsample)
for (subject, patch), grp in df.groupby(['subject_name', 'patch_name']):
    grp = grp.sort_values('block_start')
    total_n = sum(len(a) for a in grp.wheel_cumsum_distance_travelled)

    times = np.empty(total_n, dtype='datetime64[ns]')
    dists = np.empty(total_n, dtype=float)
    idx, offset = 0, 0.0

    for bs_val, arr in zip(grp.block_start, grp.wheel_cumsum_distance_travelled):
        arr = np.asarray(arr)
        n = arr.size
        offs = (np.arange(n) * dt_seconds * 1e9).astype('timedelta64[ns]')
        times[idx:idx+n] = np.datetime64(bs_val) + offs
        dists[idx:idx+n] = arr + offset
        offset += arr[-1]
        idx += n

    # Downsample by change > 0.5
    diffs = np.abs(np.diff(dists, prepend=dists[0]))
    mask = diffs > 0.5
    mask[0] = True

    fig.add_trace(
        go.Scatter(
            x=times[mask],
            y=dists[mask] / 100,  # convert to meters
            mode='lines',
            name=f"{subject} — {patch}",
            line=dict(width=1.5, color="#555555")
        )
    )

# Styling (match px.timeline aesthetic)
fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    showline=False,
    ticks='',
    showticklabels=False
)

fig.update_yaxes(
    title_text="Distance spun on wheel (m)",
    showgrid=False,
    zeroline=False,
    showline=False,
    ticks='',
    showticklabels=True
)

fig.update_layout(
    template="simple_white",
    plot_bgcolor="white",
    margin=dict(l=150, r=20, t=20, b=20),
    height=max(100, len(subjects) * 25 + 50),
    showlegend=True
)

fig.show(config={'staticPlot': True})

In [ ]:
def load_foraging_bouts(
    key: Dict[str, str],
    period_start: str,
    period_end: str
) -> pd.DataFrame:
    """Loads foraging bout data for blocks falling within a specified time period.

    Args:
        key (dict): Key to identify experiment data (e.g., {"experiment_name": "Exp1"}).
        period_start (str): Start datetime of the time period (format: '%Y-%m-%d %H:%M:%S').
        period_end (str): End datetime of the time period (format: '%Y-%m-%d %H:%M:%S').

    Returns:
        pd.DataFrame: Concatenated dataframe of foraging bouts for all matching blocks.
                      Returns an empty dataframe with predefined columns if no data found.
    """
    # Fetch block start times within the specified period
    blocks = (
        Block
        & key
        & f"block_start >= '{period_start}'"
        & f"block_end <= '{period_end}'"
    ).fetch("block_start")

    # Retrieve foraging bouts for each block
    bouts = []
    for block_start in blocks:
        block_key = key | {"block_start": str(block_start)}
        bouts.append(get_foraging_bouts(block_key, min_pellets=1))

    # Return concatenated DataFrame or empty fallback
    if bouts:
        return pd.concat(bouts, ignore_index=True)
    else:
        return pd.DataFrame(
            columns=["start", "end", "n_pellets", "cum_wheel_dist", "subject"]
        )

In [ ]:
# Create a dictionary to hold foraging data for each experiment and period
foraging_data_dict = {}

for exp in experiments:
    key = {"experiment_name": exp["name"]}
    
    # Initialize nested dictionary for this experiment
    foraging_data_dict[exp["name"]] = {}
    
    # Define periods
    periods = {
        "presocial": (exp["presocial_start"], exp["presocial_end"]),
        "social": (exp["social_start"], exp["social_end"]),
        "postsocial": (exp["postsocial_start"], exp["postsocial_end"])
    }
    
    # Load data for each period
    for period_name, (period_start, period_end) in periods.items():
        period_start = datetime.strptime(period_start, "%Y-%m-%d %H:%M:%S")
        period_end = datetime.strptime(period_end, "%Y-%m-%d %H:%M:%S")
        
        # Load foraging data for this period
        foraging_df = load_foraging_bouts(key, period_start, period_end)
        
        # Add experiment name as a column if not already present
        if 'experiment_name' not in foraging_df.columns:
            foraging_df.insert(0, "experiment_name", exp["name"])
            
        # Add period column for reference
        foraging_df['period'] = period_name
        
        # Store the data
        foraging_data_dict[exp["name"]][period_name] = foraging_df

In [ ]:
# TODO: Check this works
# TODO: Are there columns we need to explain/units we need to clarify?
exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Extract the foraging DataFrame
df = foraging_data_dict[exp_name][period]

# Display the first few rows
print(f"Foraging Data — {exp_name}, {period}")
display(df.head())

In [ ]:
# TODO: Check this works
exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Get the foraging data
df = foraging_data_dict[exp_name][period]

# Ensure correct types
df['start'] = pd.to_datetime(df['start'])
df['end'] = pd.to_datetime(df['end'])

# Plot
subjects = sorted(df['subject'].unique())

fig = px.timeline(
    df,
    x_start="start",
    x_end="end",
    y="subject",
    hover_data=["n_pellets", "cum_wheel_dist"],
    category_orders={"subject": subjects}
)

# Styling
fig.update_traces(
    opacity=1,
    marker_color="#555555",
    marker_line_color="#555555",
    marker_line_width=1.5
)

fig.update_layout(
    template='simple_white',
    plot_bgcolor='white',
    margin=dict(l=150, r=20, t=20, b=20),
    height=max(100, len(subjects) * 25 + 50),
    xaxis=dict(
        showgrid=False, zeroline=False,
        showline=False, ticks='', showticklabels=False
    ),
    yaxis=dict(
        showgrid=False, zeroline=False,
        showline=False, ticks='', showticklabels=True,
        title=''
    )
)

fig.show()

## RFID data

In [ ]:
def load_rfid_events(
    key: Dict[str, str],
    period_start: str,
    period_end: str
) -> pd.DataFrame:
    """Loads RFID events data for chunks falling within a specified time period.

    Args:
        key (dict): Key to identify experiment data (e.g., {"experiment_name": "Exp1"}).
        period_start (str): Start datetime of the time period (format: '%Y-%m-%d %H:%M:%S').
        period_end (str): End datetime of the time period (format: '%Y-%m-%d %H:%M:%S').

    Returns:
        pd.DataFrame: DataFrame containing RFID events for the specified period.
                      Returns an empty dataframe with predefined columns if no data found.
    """
    # Fetch RFID events within the specified period
    rfid_events_df = (
        streams.RfidReader 
        * streams.RfidReaderRfidEvents
        & key 
        & f'chunk_start >= "{period_start}"' 
        & f'chunk_start <= "{period_end}"'
    ).fetch(format="frame")
    
    if rfid_events_df.empty:
        # Return empty DataFrame with expected columns if no data found
        return pd.DataFrame(
            columns=["experiment_name", "chunk_start", "rfid_reader_name", "sample_count", 
                     "timestamps", "rfid"]
        )
    
    # Get subject details for RFID mapping
    subject_detail = subject.SubjectDetail.fetch(format="frame")
    subject_detail.reset_index(inplace=True)
    
    # Create mapping from RFID to subject ID
    rfid_to_lab_id = dict(zip(subject_detail['lab_id'], subject_detail['subject']))
    
    rfid_events_df['rfid'] = [
        [rfid_to_lab_id.get(str(rfid)) for rfid in rfid_array] 
        for rfid_array in rfid_events_df['rfid']
    ]
    
    # Extract experiment_name and chunk_start from the index before resetting
    rfid_events_df['experiment_name'] = [idx[0] for idx in rfid_events_df.index]
    rfid_events_df['chunk_start'] = [idx[3] for idx in rfid_events_df.index]  # Assuming chunk_start is at index 3
    
    # Reset the index and drop the index column
    rfid_events_df = rfid_events_df.reset_index(drop=True)
    
    # Reorder columns to put experiment_name first and chunk_start second
    cols = ['experiment_name', 'chunk_start'] + [col for col in rfid_events_df.columns if col not in ['experiment_name', 'chunk_start']]
    rfid_events_df = rfid_events_df[cols]
    
    return rfid_events_df

In [ ]:
# Create a dictionary to hold RFID data for each experiment and period
rfid_data_dict = {}

for exp in experiments:
    key = {"experiment_name": exp["name"]}
    exp_name = exp["name"]

    # Initialize nested dictionary for this experiment
    rfid_data_dict[exp_name] = {}

    # Define periods
    periods = {
        "presocial": (exp["presocial_start"], exp["presocial_end"]),
        "social": (exp["social_start"], exp["social_end"]),
        "postsocial": (exp["postsocial_start"], exp["postsocial_end"])
    }

    # Load data for each period
    for period_name, (period_start, period_end) in periods.items():
        # Handle datetime formatting
        if not isinstance(period_start, str):
            period_start_str = period_start.strftime("%Y-%m-%d %H:%M:%S")
        else:
            period_start_str = period_start

        if not isinstance(period_end, str):
            period_end_str = period_end.strftime("%Y-%m-%d %H:%M:%S")
        else:
            period_end_str = period_end

        # Load RFID data
        rfid_df = load_rfid_events(key, period_start_str, period_end_str)

        # Add metadata columns
        if not rfid_df.empty:
            if 'experiment_name' not in rfid_df.columns:
                rfid_df.insert(0, "experiment_name", exp_name)
            rfid_df['period'] = period_name

            # Clean for aeon3 presocial only 
            # TODO: Check this works
            if exp_name == "social0.2-aeon3" and period_name == "presocial":
                rfid_df['rfid_reader_name'] = rfid_df['rfid_reader_name'].replace({
                    'Patch1Rfid': 'TEMP_PLACEHOLDER',
                    'Patch2Rfid': 'Patch1Rfid'
                })
                rfid_df['rfid_reader_name'] = rfid_df['rfid_reader_name'].replace({
                    'TEMP_PLACEHOLDER': 'Patch2Rfid'
                })
                print("Cleaned reader names for aeon3 presocial:", rfid_df["rfid_reader_name"].unique())

        # Store the cleaned DataFrame
        rfid_data_dict[exp_name][period_name] = rfid_df

In [ ]:
# TODO: Check this works
# TODO: Are there columns we need to explain/units we need to clarify?
exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Extract the RFID DataFrame
df = rfid_data_dict[exp_name][period]

# Display the first few rows
print(f"RFID Data — {exp_name}, {period}")
display(df.head())

In [ ]:
# TODO: add example plots/uses for the rfid_data DataFrames
# We may want to give them the precise location of the RFID readers? 
# I get them from the api but maybe we can give them a premade hardcoded dict
# All my code that uses the RFID df is quite long because it relies on first matching RFID reads to SLEAP positions, and then further calculations from there
# Is there anything simpler we could do here instead?

## Position data

In [ ]:
def load_position_data(
    key: Dict[str, str],
    period_start: str,
    period_end: str
) -> pd.DataFrame:
    """Loads position data (centroid tracking) for a specified time period.

    Args:
        key (dict): Key to identify experiment data (e.g., {"experiment_name": "Exp1"}).
        period_start (str): Start datetime of the time period.
        period_end (str): End datetime of the time period.

    Returns:
        pd.DataFrame: DataFrame containing position data for the specified period.
                     Returns an empty DataFrame if no data found.
    """
    try:
        print(f"  Querying data from {period_start} to {period_end}...")

        # Create chunk restriction for the time period
        chunk_restriction = acquisition.create_chunk_restriction(
            key["experiment_name"], period_start, period_end
        )

        # Create the query
        pose_query = (
            streams.SpinnakerVideoSource
            * tracking.SLEAPTracking.PoseIdentity.proj(
                "identity_name", "identity_likelihood", "anchor_part"
            )
            * tracking.SLEAPTracking.AnchorPart
            & key
            & {
                "spinnaker_video_source_name": "CameraTop",
            }
            & chunk_restriction
        )

        # Fetch the data
        centroid_df = fetch_stream(pose_query)

        # Clean up the dataframe
        if not centroid_df.empty:
            if "spinnaker_video_source_name" in centroid_df.columns:
                centroid_df.drop(columns=["spinnaker_video_source_name"], inplace=True)

            # Add experiment name column for reference
            centroid_df.insert(0, "experiment_name", key["experiment_name"])

            print(f"  Retrieved {len(centroid_df)} rows of position data")
        else:
            print("  No data found for the specified period")

        return centroid_df

    except Exception as e:
        print(
            f"  Error loading position data for {key['experiment_name']} ({period_start} "
            f"to {period_end}): {e}"
        )
        return pd.DataFrame()

In [ ]:
# Create a dictionary to hold position data for each experiment and period
position_data_dict = {}

for exp in experiments:
    key = {"experiment_name": exp["name"]}

    # Initialize nested dictionary for this experiment
    position_data_dict[exp["name"]] = {}

    # Define periods
    periods = {
        "presocial": (exp["presocial_start"], exp["presocial_end"]),
        "social": (exp["social_start"], exp["social_end"]),
        "postsocial": (exp["postsocial_start"], exp["postsocial_end"])
    }

    # Load data for each period
    for period_name, (period_start, period_end) in periods.items():
        print(f"  Loading {period_name} period...")

        period_start = datetime.strptime(period_start, "%Y-%m-%d %H:%M:%S")
        period_end = datetime.strptime(period_end, "%Y-%m-%d %H:%M:%S")

        # Load position data for this period
        position_df = load_position_data(key, period_start, period_end)
        position_df.reset_index(inplace=True)

        # Add period column for reference if not empty
        if not position_df.empty:
            position_df['period'] = period_name

        # Store the data
        position_data_dict[exp["name"]][period_name] = position_df

        # Print data size info
        if not position_df.empty:
            memory_usage_mb = position_df.memory_usage(deep=True).sum() / (1024 * 1024)
            print(f"    {period_name}: {len(position_df)} rows, {memory_usage_mb:.2f} MB in memory")
        else:
            print(f"    {period_name}: No data available")

In [ ]:
# TODO: Check this works
# TODO: Are there columns we need to explain/units we need to clarify?
exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Extract the position DataFrame
df = position_data_dict[exp_name][period]

# Display the first few rows
print(f"Position Data — {exp_name}, {period}")
display(df.head())

In [ ]:
# TODO: Check this works
exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Load and clean the data
df = position_data_dict[exp_name][period]
df = clean_swaps(df)  # TODO: check this is necessary, hopefully already ingested in DJ
df = df.dropna(subset=['x', 'y']).reset_index()

# Add a 'date' column for grouping by day
timestamp_col = df.columns[0]
df['date'] = pd.to_datetime(df[timestamp_col]).dt.date

# Define bin size and grid
bin_size = 25  # pixels
x_bins = np.arange(df['x'].min(), df['x'].max() + bin_size, bin_size)
y_bins = np.arange(df['y'].min(), df['y'].max() + bin_size, bin_size)

# Plot a heatmap for each day
for day, day_df in df.groupby('date'):
    heatmap, x_edges, y_edges = np.histogram2d(
        day_df['x'],
        day_df['y'],
        bins=[x_bins, y_bins]
    )

    fig = go.Figure(data=go.Heatmap(
        z=heatmap.T,  # transpose to align axes correctly
        x=x_edges[:-1],
        y=y_edges[:-1],
        colorscale='Greys',
        colorbar=dict(title='Count'),
        showscale=True
    ))

    fig.update_layout(
        title=f"Position Heatmap — {exp_name}, {period} ({day})",
        xaxis=dict(
            title='x (pixels)',
            showgrid=False,
            showticklabels=False,
            zeroline=False
        ),
        yaxis=dict(
            title='y (pixels)',
            showgrid=False,
            showticklabels=False,
            zeroline=False,
            scaleanchor='x'
        ),
        template='simple_white',
        plot_bgcolor='white',
        margin=dict(l=40, r=40, t=40, b=40),
        width=500,
        height=500
    )

    fig.show()